In [ ]:
# Cell 0: Install quantization and LLM libraries
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn pandas

In [ ]:
# Cell 1: Authenticate with Hugging Face to access Llama 3
from huggingface_hub import notebook_login

print("Please paste your Hugging Face Access Token below:")
notebook_login()

Please paste your Hugging Face Access Token below:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Cell 2: Data Preparation
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd
import re
import string

print("Loading test data...")
ds_cb = concatenate_datasets([load_dataset("SALT-NLP/CultureBank", split='tiktok'),
                              load_dataset("SALT-NLP/CultureBank", split='reddit')])
ds_mnli = load_dataset("multi_nli", split="train")

ds_cb = ds_cb.rename_column("actor_behavior", "text").select_columns(["text"])
ds_mnli = ds_mnli.rename_column("premise", "text").select_columns(["text"])

# Filter MNLI for length
mnli_df = ds_mnli.to_pandas()
mnli_df['word_count'] = mnli_df['text'].apply(lambda x: len(str(x).split()))
long_mnli_df = mnli_df[mnli_df['word_count'] > 8].drop_duplicates(subset=["text"])
ds_generic = Dataset.from_pandas(long_mnli_df, preserve_index=False).remove_columns(["word_count"])

# Universal Sanitizer
def sanitize_and_debias(example):
    text = str(example["text"])
    words_to_remove = r'\b(customary|common practice|norm|culture|tradition|expected)\b'
    text = re.sub(words_to_remove, '', text, flags=re.IGNORECASE)
    location_pattern = r'^In\s+[A-Z][a-z]+\s*(culture|society)?\s*,'
    text = re.sub(location_pattern, '', text)
    example["text"] = text.rstrip(string.punctuation).strip()
    example["text"] = re.sub(r'\s+', ' ', example["text"])
    return example

# Grab 50 of each for a quick but robust 100-sentence test
test_norms = ds_cb.shuffle(seed=42).select(range(50)).map(sanitize_and_debias)
test_generics = ds_generic.shuffle(seed=42).select(range(50)).map(sanitize_and_debias)

test_sentences = list(test_norms['text']) + list(test_generics['text'])
true_labels = ["Norm"] * 50 + ["Generic"] * 50

print(f"Data ready! {len(test_sentences)} sentences prepared for Llama.")

Loading test data...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Data ready! 100 sentences prepared for Llama.


In [ ]:
# Cell 3: Load Llama 3 in 4-Bit Precision
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

print("Loading Llama-3-8B-Instruct (This will take a few minutes)...")

# Configure the 4-bit compression
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama doesn't have a default pad token, so we set it to the End-Of-Sentence token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Build the pipeline
llama_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=10, # We only need 1 word back
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)
print("Llama 3 is loaded and ready!")

Loading Llama-3-8B-Instruct (This will take a few minutes)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
# Cell 4: Advanced Chat-Templated Prompts

def get_zero_shot_messages(text):
    return [
        {"role": "system", "content": """You are an expert sociologist.
A "Norm" is a prescriptive behavioral rule, social expectation, or custom (e.g., taking shoes off, tipping).
A "Generic" is a plain scientific, geographical, or observational fact.
First, write a 1-sentence reasoning. Then, you MUST end your response exactly with "LABEL: Norm" or "LABEL: Generic"."""},
        {"role": "user", "content": f"Classify this sentence: '{text}'"}
    ]

def get_few_shot_messages(text):
    return [
        {"role": "system", "content": """You are an expert sociologist.
A "Norm" is a prescriptive behavioral rule, social expectation, or custom.
A "Generic" is a plain scientific, geographical, or observational fact.
First, write a 1-sentence reasoning. Then, you MUST end your response exactly with "LABEL: Norm" or "LABEL: Generic"."""},
        {"role": "user", "content": "Classify this sentence: 'Water boils at 100 degrees Celsius.'"},
        {"role": "assistant", "content": "This is a statement about a universal scientific property of physics, not human behavior. LABEL: Generic"},
        {"role": "user", "content": "Classify this sentence: 'You should remove your shoes before entering the house.'"},
        {"role": "assistant", "content": "This describes a cultural expectation and rule for polite behavior in many societies. LABEL: Norm"},
        {"role": "user", "content": f"Classify this sentence: '{text}'"}
    ]

print("Advanced CoT Prompts constructed successfully.")

In [ ]:
# Cell 5: Upgraded Evaluation Engine
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm
import re

zero_shot_preds = []
few_shot_preds = []

# We need a robust parser to hunt for the exact label at the end of Llama's reasoning
def extract_label(response_data):
    # Safety catch: if the pipeline does return a dictionary format, extract the string
    if isinstance(response_data, list):
        response_text = response_data[-1]['content']
    else:
        response_text = str(response_data)

    # Searches for 'LABEL: Norm' or 'LABEL: Generic' ignoring case
    match = re.search(r"LABEL:\s*(Norm|Generic)", response_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Generic" # Safe fallback

print("Running Llama 3 CoT Evaluation...")
for text in tqdm(test_sentences, desc="Classifying 100 sentences"):

    # 1. Test Zero-Shot
    zs_messages = get_zero_shot_messages(text)
    # Get the generated output directly
    zs_output = llama_pipe(zs_messages, max_new_tokens=60)[0]['generated_text']
    zero_shot_preds.append(extract_label(zs_output))

    # 2. Test Few-Shot
    fs_messages = get_few_shot_messages(text)
    # Get the generated output directly
    fs_output = llama_pipe(fs_messages, max_new_tokens=60)[0]['generated_text']
    few_shot_preds.append(extract_label(fs_output))


# --- Print Results ---
print("\n" + "="*50)
print(" 🚀 LLAMA 3 ADVANCED ABLATION RESULTS 🚀 ")
print("="*50)

print("\n--- ZERO-SHOT PERFORMANCE (WITH CoT) ---")
zs_acc = accuracy_score(true_labels, zero_shot_preds)
print(f"Accuracy: {zs_acc:.2%}")
print(classification_report(true_labels, zero_shot_preds, target_names=["Generic", "Norm"]))

print("\n--- FEW-SHOT PERFORMANCE (WITH CoT) ---")
fs_acc = accuracy_score(true_labels, few_shot_preds)
print(f"Accuracy: {fs_acc:.2%}")
print(classification_report(true_labels, few_shot_preds, target_names=["Generic", "Norm"]))

In [ ]:
# Cell 6: Manual Interactive Testing
import textwrap

def test_custom_sentence(sentence):
    print(f"INPUT SENTENCE: '{sentence}'")
    print("=" * 60)

    # --- Zero-Shot Execution ---
    zs_msgs = get_zero_shot_messages(sentence)
    zs_raw_output = llama_pipe(zs_msgs, max_new_tokens=60)[0]['generated_text']

    # Handle the output format gracefully
    if isinstance(zs_raw_output, list):
        zs_text = zs_raw_output[-1]['content'].strip()
    else:
        zs_text = str(zs_raw_output).strip()

    print("ZERO-SHOT REASONING:")
    print(textwrap.fill(zs_text, width=60))
    print("-" * 60)

    # --- Few-Shot Execution ---
    fs_msgs = get_few_shot_messages(sentence)
    fs_raw_output = llama_pipe(fs_msgs, max_new_tokens=60)[0]['generated_text']

    if isinstance(fs_raw_output, list):
        fs_text = fs_raw_output[-1]['content'].strip()
    else:
        fs_text = str(fs_raw_output).strip()

    print("FEW-SHOT REASONING:")
    print(textwrap.fill(fs_text, width=60))
    print("=" * 60 + "\n")

# --- Enter your custom sentences here! ---
my_test_sentences = [
    "Tipping 20 percent at sit-down restaurants is considered polite.",
    "A triangle has exactly three sides and three angles.",
    "Men usually take off their hats when entering a church.",
    "Water must reach 100 degrees Celsius to boil."
]

for text in my_test_sentences:
    test_custom_sentence(text)

In [ ]:
# Cell 7: Install and Authenticate OpenAI
!pip install -q openai

import getpass
import os
from openai import OpenAI

print("Please securely paste your OpenAI API Key below:")
os.environ["OPENAI_API_KEY"] = getpass.getpass()

# Initialize the OpenAI Client
client = OpenAI()
print("OpenAI Client connected!")

In [ ]:
# Cell 8: Define the Chat Prompts for GPT

def get_gpt_zero_shot(text):
    return [
        {"role": "system", "content": """You are an expert sociologist.
A "Norm" is a prescriptive behavioral rule, social expectation, or custom (e.g., taking shoes off, tipping).
A "Generic" is a plain scientific, geographical, or observational fact.
First, write a 1-sentence reasoning. Then, you MUST end your response exactly with "LABEL: Norm" or "LABEL: Generic"."""},
        {"role": "user", "content": f"Classify this sentence: '{text}'"}
    ]

def get_gpt_few_shot(text):
    return [
        {"role": "system", "content": """You are an expert sociologist.
A "Norm" is a prescriptive behavioral rule, social expectation, or custom.
A "Generic" is a plain scientific, geographical, or observational fact.
First, write a 1-sentence reasoning. Then, you MUST end your response exactly with "LABEL: Norm" or "LABEL: Generic"."""},
        {"role": "user", "content": "Classify this sentence: 'Water boils at 100 degrees Celsius.'"},
        {"role": "assistant", "content": "This is a statement about a universal scientific property of physics, not human behavior. LABEL: Generic"},
        {"role": "user", "content": "Classify this sentence: 'You should remove your shoes before entering the house.'"},
        {"role": "assistant", "content": "This describes a cultural expectation and rule for polite behavior in many societies. LABEL: Norm"},
        {"role": "user", "content": f"Classify this sentence: '{text}'"}
    ]

print("GPT Prompts ready.")

In [ ]:
# Cell 9: Run the OpenAI Evaluation
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm
import re
import time

gpt_zero_shot_preds = []
gpt_few_shot_preds = []

def extract_gpt_label(response_text):
    match = re.search(r"LABEL:\s*(Norm|Generic)", response_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "Generic"

print("Pinging OpenAI API for 100 sentences...")

for text in tqdm(test_sentences, desc="Evaluating with GPT-4o-mini"):
    try:
        # 1. Zero-Shot API Call
        zs_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=get_gpt_zero_shot(text),
            max_tokens=60,
            temperature=0.0 # Temperature 0 makes it deterministic and strict
        )
        zs_output = zs_response.choices[0].message.content
        gpt_zero_shot_preds.append(extract_gpt_label(zs_output))

        # 2. Few-Shot API Call
        fs_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=get_gpt_few_shot(text),
            max_tokens=60,
            temperature=0.0
        )
        fs_output = fs_response.choices[0].message.content
        gpt_few_shot_preds.append(extract_gpt_label(fs_output))

    except Exception as e:
        print(f"API Error on sentence: '{text}'. Error: {e}")
        # Append safe fallbacks so the arrays don't misalign
        gpt_zero_shot_preds.append("Generic")
        gpt_few_shot_preds.append("Generic")
        # Small sleep to recover from rate limits
        time.sleep(2)

# --- Print Results ---
print("\n" + "="*50)
print(" 🚀 OPENAI GPT-4o-mini ABLATION RESULTS 🚀 ")
print("="*50)

print("\n--- ZERO-SHOT PERFORMANCE ---")
zs_acc = accuracy_score(true_labels, gpt_zero_shot_preds)
print(f"Accuracy: {zs_acc:.2%}")
print(classification_report(true_labels, gpt_zero_shot_preds, target_names=["Generic", "Norm"]))

print("\n--- FEW-SHOT PERFORMANCE ---")
fs_acc = accuracy_score(true_labels, gpt_few_shot_preds)
print(f"Accuracy: {fs_acc:.2%}")
print(classification_report(true_labels, gpt_few_shot_preds, target_names=["Generic", "Norm"]))